# Lab 11 — Deep reinforcement learning with DQN

**Environment:** Gymnasium `CartPole-v1`.

Compare a stabilised DQN with two ablations: no replay and no target network. The scientific question is whether the stabilizers improve learning reliability, not whether one lucky run reaches a high score.

**Evaluation:** 20 fresh episodes per saved condition, greedy policy.

In [ ]:
!pip -q install gymnasium torch pandas
import gymnasium as gym, numpy as np, pandas as pd, torch
from torch import nn
env=gym.make('CartPole-v1'); print(env.observation_space,env.action_space); env.close()

## 1. Prediction

Predict that replay should reduce correlation between consecutive updates and that a target network should reduce moving-target instability. Therefore removing either component should increase variance or reduce learning speed in at least some seeds.

In [ ]:
class Q(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Linear(4,64),nn.ReLU(),nn.Linear(64,2))
    def forward(self,x): return self.net(x)

def train_condition(use_replay=True,use_target=True,seed=0,episodes=80):
    rng=np.random.default_rng(seed); env=gym.make('CartPole-v1'); q=Q(); target=Q(); target.load_state_dict(q.state_dict()); opt=torch.optim.Adam(q.parameters(),1e-3); replay=[]; rewards=[]
    for ep in range(episodes):
        s,_=env.reset(seed=seed+ep); total=0
        for _ in range(500):
            eps=max(.05,1-ep/60); a=env.action_space.sample() if rng.random()<eps else int(q(torch.tensor(s,dtype=torch.float32)).argmax())
            ns,r,term,trunc,_=env.step(a); replay.append((s,a,r,ns,term)); replay=replay[-2000:]; total+=r; s=ns
            if len(replay)>=32:
                batch=[replay[i] for i in rng.integers(0,len(replay),32)] if use_replay else replay[-32:]
                ss=torch.tensor(np.array([b[0] for b in batch]),dtype=torch.float32); aa=torch.tensor([b[1] for b in batch]); rr=torch.tensor([b[2] for b in batch]); nnx=torch.tensor(np.array([b[3] for b in batch]),dtype=torch.float32); dd=torch.tensor([b[4] for b in batch],dtype=torch.float32)
                ref=target if use_target else q; y=rr+.99*(1-dd)*ref(nnx).max(1).values.detach(); pred=q(ss).gather(1,aa[:,None]).squeeze(); loss=((pred-y)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            if term or trunc: break
        if use_target and ep%10==0: target.load_state_dict(q.state_dict())
        rewards.append(total)
    env.close(); return q,np.asarray(rewards)

def evaluate(q,seed=1000,n=20):
    env=gym.make('CartPole-v1'); scores=[]
    for e in range(n):
        s,_=env.reset(seed=seed+e); total=0
        for _ in range(500):
            a=int(q(torch.tensor(s,dtype=torch.float32)).argmax()); s,r,t,tr,_=env.step(a); total+=r
            if t or tr: break
        scores.append(total)
    env.close(); return np.mean(scores),np.std(scores)

rows=[]
for name,rep,tgt in [('full',True,True),('no_replay',False,True),('no_target',True,False)]:
    q,rewards=train_condition(rep,tgt,seed=3); mean,std=evaluate(q)
    rows.append([name,float(rewards[-20:].mean()),mean,std])
results=pd.DataFrame(rows,columns=['condition','training_return_last20','eval_mean_return','eval_std']); results

## Conclusion / answer key

The defensible result is **mean ± standard deviation over fresh evaluation episodes**, plus the training curve. If one ablation wins once, repeat seeds before claiming an algorithmic difference.

**Research extension:** run 5 seeds per condition and report time-to-threshold, sample efficiency, evaluation variance, and failure episodes.